# 🐦‍⬛ Karga-2B-Thinking — Çalışan Notebook
Türkçe düşünen, adım adım çözen küçük model.

In [5]:
# 1. Gereksinimleri Yükle
!pip install -q transformers accelerate torch

In [6]:
# 2. Model ve Tokenizer'ı Yükle
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "ilkayO/Karga-2B-Thinking"

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Model yükleniyor (bu biraz sürebilir)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = next(model.parameters()).device
print(f"✅ Model hazır! Cihaz: {device}")

Tokenizer yükleniyor...
Model yükleniyor (bu biraz sürebilir)...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

✅ Model hazır! Cihaz: cuda:0


In [7]:
# 3. Yardımcı Fonksiyon: Yanıt Üret

def karga_sor(soru: str, max_new_tokens: int = 1024) -> str:
    messages = [
        {"role": "system", "content": "Adın Karga. Soruları mantıklı ve adım adım düşünerek yanıtla."},
        {"role": "user", "content": soru}
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.get_vocab():
        terminators.append(tokenizer.convert_tokens_to_ids("<|eot_id|>"))

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id
        )

    new_ids = output_ids[0][inputs.input_ids.shape[-1]:]
    full_response = tokenizer.decode(new_ids, skip_special_tokens=True)
    return full_response


def yaniti_goster(soru: str):
    """
    Soruyu sor, düşünce ve cevabı güzel formatta yazdır.
    """
    print(f"\n❓ Soru: {soru}")
    print("-" * 60)

    yanit = karga_sor(soru)

    if "</think>" in yanit:
        parcalar = yanit.split("</think>")
        dusunce = parcalar[0].replace("<think>", "").strip()
        cevap = parcalar[1].strip() if len(parcalar) > 1 else ""

        print("\n💭 Düşünce Süreci:")
        print(dusunce)
        print("\n✅ Cevap:")
        print(cevap)
    else:
        print("\n✅ Yanıt:")
        print(yanit)

    return yanit

print("✅ Fonksiyonlar hazır!")

✅ Fonksiyonlar hazır!


In [33]:
# 4. Test 1: Matematik
yaniti_goster("Ali'nin 10 lirası vardır. Parasını harcamak istiyordur. Ali 10 lirasıyla 4 liralık bir defter satın almıştır. Geriye kaç lirası kaldı? Adım adım hesapla.")


❓ Soru: Ali'nin 10 lirası vardır. Parasını harcamak istiyordur. Ali 10 lirasıyla 4 liralık bir defter satın almıştır. Geriye kaç lirası kaldı? Adım adım hesapla.
------------------------------------------------------------

💭 Düşünce Süreci:
1. Ali, 10 lirası ile 4 liralık bir defter aldı.
2. Defterin fiyatı 10 liradır.
3. Ali'nin kalan parası 10 - 4 = <<10-4=6>>6 liradır.

✅ Cevap:
6


"1. Ali, 10 lirası ile 4 liralık bir defter aldı.\n2. Defterin fiyatı 10 liradır.\n3. Ali'nin kalan parası 10 - 4 = <<10-4=6>>6 liradır.\n</think>\n6"

In [27]:
# 5. Test 2: Mantık
yaniti_goster("Bir kedi, bir köpek ve bir masa. Bunlardan hangisi cansız bir varlıktır? Nedenini açıkla.")


❓ Soru: Bir kedi, bir köpek ve bir masa. Bunlardan hangisi cansız bir varlıktır? Nedenini açıkla.
------------------------------------------------------------

💭 Düşünce Süreci:
Soru, cansız varlıkları tanımlamasını istiyor. 'Cansız' (inert) terimi, maddenin kimyasal yapısını değiştirmeyen, reaksiyona girmeyen veya yok olan varlıkları ifade eder. 

Doğru cevap olarak: Masa.

✅ Cevap:
Masa.


"Soru, cansız varlıkları tanımlamasını istiyor. 'Cansız' (inert) terimi, maddenin kimyasal yapısını değiştirmeyen, reaksiyona girmeyen veya yok olan varlıkları ifade eder. \n\nDoğru cevap olarak: Masa.\n</think>\nMasa."

In [64]:
# 6. Test 3: Kodlama
yaniti_goster("İki sayıyı toplayan 'sum_two' adında bir Python fonksiyonu yazın.")


❓ Soru: İki sayıyı toplayan 'sum_two' adında bir Python fonksiyonu yazın.
------------------------------------------------------------

💭 Düşünce Süreci:
1. İki tam sayı listesi tanımlayın (n = [1, 2, 3]).
2. Toplama işlemi için '+' operatörünü kullanın.
3. İki sayıyı birleştirmek için toplam işareti (*) ekleyin.
4. Döngü sonucu döndürün.
5. Kodun çalıştığından emin olmak için bir test yazın ve sonucu yazdırın.

✅ Cevap:
```python
def sum_two(n):
    result = n + 1
    # Toplamı bulmak için toplam işareti kullan
    return result

# Test için örnek: 1 ve 2'yi toplayalım
print("{} ve {} toplandı!".format(1, 2))
# Çıktı: {1, 2}
```


'1. İki tam sayı listesi tanımlayın (n = [1, 2, 3]).\n2. Toplama işlemi için \'+\' operatörünü kullanın.\n3. İki sayıyı birleştirmek için toplam işareti (*) ekleyin.\n4. Döngü sonucu döndürün.\n5. Kodun çalıştığından emin olmak için bir test yazın ve sonucu yazdırın.\n</think>\n```python\ndef sum_two(n):\n    result = n + 1\n    # Toplamı bulmak için toplam işareti kullan\n    return result\n\n# Test için örnek: 1 ve 2\'yi toplayalım\nprint("{} ve {} toplandı!".format(1, 2))\n# Çıktı: {1, 2}\n```'

In [62]:
# 7. (Opsiyonel) Streaming ile Gerçek Zamanlı Yanıt
from transformers import TextIteratorStreamer
from threading import Thread

def karga_streaming(soru: str):
    messages = [
        {"role": "system", "content": "Adın Karga. Soruları mantıklı ve adım adım düşünerek yanıtla."},
        {"role": "user", "content": soru}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.get_vocab():
        terminators.append(tokenizer.convert_tokens_to_ids("<|eot_id|>"))

    generation_kwargs = {
        "input_ids": inputs.input_ids,
        "attention_mask": inputs.attention_mask,
        "streamer": streamer,
        "max_new_tokens": 1024,
        "temperature": 0.6,
        "top_p": 0.9,
        "repetition_penalty": 1.1,
        "do_sample": True,
        "eos_token_id": terminators,
        "pad_token_id": tokenizer.eos_token_id
    }

    print(f"\n❓ Soru: {soru}")
    print("-" * 60)

    Thread(target=model.generate, kwargs=generation_kwargs).start()

    tam_yanit = ""
    for token in streamer:
        print(token, end="", flush=True)
        tam_yanit += token

    print()
    return tam_yanit

# Streaming testi
karga_streaming("Elma bir meyve midir? Sebebini açıkla.")


❓ Soru: Elma bir meyve midir? Sebebini açıkla.
------------------------------------------------------------
1. Soru, elmanın meyvemsi özelliklerini ve kökenini sorgulamaktadır.
2. Elma (Pear), Rosales takımına ait bir ağaç bitkisidir.
3. Elmalar genellikle tatlı ve sulu meyvelerdir (örneğin elma, şeftali, armut).
4. Kökeni Güney Amerika'dır (özellikle And Dağları) ve Avrupa'ya bu bölgeden yayılmıştır.
5. Bu bilgi doğrulanabilir ve mantıklıdır.
</think>
Evet, elma bir meyvedir.


"1. Soru, elmanın meyvemsi özelliklerini ve kökenini sorgulamaktadır.\n2. Elma (Pear), Rosales takımına ait bir ağaç bitkisidir.\n3. Elmalar genellikle tatlı ve sulu meyvelerdir (örneğin elma, şeftali, armut).\n4. Kökeni Güney Amerika'dır (özellikle And Dağları) ve Avrupa'ya bu bölgeden yayılmıştır.\n5. Bu bilgi doğrulanabilir ve mantıklıdır.\n</think>\nEvet, elma bir meyvedir."